# 빌드 캐시와 이미지 크기

Docker Desktop의 실습 전용 이미지와 스택만 사용해 결과를 확인합니다.

## 실행 환경 준비

두 Dockerfile의 빌드와 측정에 같은 명령 함수를 사용합니다.

### 이 준비 셀이 하는 일

이 준비 셀은 현재 폴더와 부모 폴더에서 `Dockerfile.baseline`이 있는 Ch32 실습 폴더를 찾습니다. `run(*args, check=True)`은 모든 Docker 명령을 그 폴더에서 실행하고 표준 출력·오류와 종료 코드를 `CompletedProcess`로 반환합니다. 실패 자체를 관찰하는 셀만 `check=False`를 전달하며, 마지막에는 Docker Engine이 응답하는지 확인합니다.

In [1]:
# Docker 명령을 일정한 폴더에서 실행하고 실패도 결과로 받습니다.
from pathlib import Path
import json, subprocess, time, urllib.request
# 챕터 폴더와 notebooks 폴더 어디에서 시작해도 제공 자료를 찾습니다.
candidates = [parent / "labs/part7/ch32-docker" for parent in (Path.cwd(), *Path.cwd().parents)]
candidates += [Path.cwd(), Path.cwd().parent]
LAB = next(path for path in candidates if (path / "Dockerfile.baseline").is_file())
def run(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(args, cwd=LAB, text=True, capture_output=True, check=check)
version = run("docker", "version", "--format", "{{.Server.Version}}")
print("Docker Engine", version.stdout.strip())
assert version.stdout.strip()

Docker Engine 29.4.1


## 두 이미지 빌드

기준 이미지와 최적화 이미지를 같은 애플리케이션 코드로 만듭니다.

### 바로 다음 코드를 읽는 방법

Docker 빌드 캐시는 명령과 그 명령이 의존하는 파일이 같을 때 이전 레이어를 재사용합니다. 어느 `COPY`가 바뀌면 그 단계뿐 아니라 뒤의 단계도 다시 실행됩니다. 그래서 자주 바뀌지 않는 의존성 잠금 파일을 먼저 복사해 설치하고, 자주 바뀌는 애플리케이션 소스는 뒤에서 복사합니다. 출력의 `CACHED`는 해당 빌드에서 재사용된 단계의 관찰값입니다.

In [2]:
# 빌드 파일만 다르고 앱과 잠금 파일은 같습니다.
run("docker", "build", "-f", "Dockerfile.baseline", "-t", "ch32-baseline:local", ".")
run("docker", "build", "-f", "Dockerfile", "-t", "ch32-optimized:local", ".")
print("two images built")

two images built


## 크기 비교

Docker가 보고한 압축 전 이미지 크기를 바이트 단위로 읽습니다.

In [3]:
# 동일한 inspect 형식으로 크기를 비교합니다.
sizes: dict[str, int] = {}
for name in ("ch32-baseline:local", "ch32-optimized:local"):
    result = run("docker", "image", "inspect", name, "--format", "{{.Size}}")
    sizes[name] = int(result.stdout)
reduction = 1 - sizes["ch32-optimized:local"] / sizes["ch32-baseline:local"]
print(sizes, "reduction", f"{reduction:.1%}")
assert reduction >= 0.5

{'ch32-baseline:local': 418111662, 'ch32-optimized:local': 57560686} reduction 86.2%


## 반복 빌드와 캐시

소스가 바뀌지 않은 상태로 다시 빌드해 CACHED 기록을 셉니다.

In [4]:
# 일반 진행 출력에서 캐시 재사용 표시를 확인합니다.
again = run("docker", "build", "--progress=plain", "-f", "Dockerfile", "-t", "ch32-optimized:local", ".")
combined = again.stdout + again.stderr
cached_count = combined.count("CACHED")
print("cached steps", cached_count)
assert cached_count >= 3

cached steps 9


## 실험 결과 해석

이번 수치는 arm64와 고정된 태그에서 얻었으며 다른 시점에는 달라질 수 있습니다.

### 이 단계에서 구분할 범위

캐시 적중 수와 이미지 크기는 서로 다른 지표입니다. 캐시가 많아도 최종 이미지가 클 수 있고, 멀티스테이지로 작은 이미지를 만들어도 첫 빌드는 오래 걸릴 수 있습니다. 비교할 때는 같은 빌드 컨텍스트와 플랫폼을 사용하고, `--no-cache`와 `--pull`이 각각 단계 재실행과 새 베이스 확인이라는 다른 역할임을 구분합니다.

In [5]:
# 두 이미지가 같은 HTTP 동작을 제공하는지도 함께 검사합니다.
for name in sizes:
    result = run("docker", "run", "--rm", name, "python", "-c", "from app import health; print(health())")
    print(name, result.stdout.strip())
    assert "'status': 'ok'" in result.stdout

ch32-baseline:local {'status': 'ok'}


ch32-optimized:local {'status': 'ok'}
